# T5Gemma 2 — CEFR classification (fine-tuning)

Fine-tunes T5Gemma 2 on Write & Improve (A2–C1). One backbone, two modes set by `cfg.mode`:

- `"nominal"` — cross-entropy over the 8 levels (the retained baseline)
- `"corn"` — rank-consistent ordinal head + CORN loss (primary condition)

The notebook imports functions from model.py, ordinal.py and metrics.py:
the CORN loss and decoding come from `coral_pytorch` (Raschka's reference package, `pip install coral_pytorch`); `model.py`
(encoder + swappable head), and **`metrics.py` — the same module the Gemma 4 notebook
imports, so both experiments are scored by identical code.

In [1]:
import os, json, csv
from dataclasses import dataclass

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding,
)

# !pip install coral_pytorch
from coral_pytorch.losses import corn_loss               # reference CORN loss
from coral_pytorch.dataset import corn_label_from_logits  # rank-consistent decoding
from model import T5Gemma2OrdinalClassifier             # encoder + mean-pool + head
from metrics import evaluate_predictions, LABEL_NAMES   # SHARED with the Gemma 4 notebook

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]   # matches Gemma 4
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}            # A2=0 ... C1+=7
NUM_CLASSES = len(LEVELS)                                      # 8


@dataclass
class CFG:
    mode: str = "corn"                          # "corn" | "nominal"
    model_id: str = "google/t5gemma-2-4b-4b"    # 270m-270m | 1b-1b | 4b-4b
    train_path: str = "data/train.jsonl"
    eval_path: str  = "data/val.jsonl"
    test_path: str  = "data/test.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    epochs: float = 4.0
    batch_size: int = 8
    grad_accum: int = 1
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    bf16: bool = True
    use_lora: bool = False                       # flip on if VRAM is tight
    lora_r: int = 16
    output_dir: str = "runs/t5gemma2_corn"
    seed: int = 42

cfg = CFG()

In [2]:
# --- data ---------------------------------------------------------------
def read_rows(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    elif ext == ".json":
        with open(path, encoding="utf-8") as f:
            yield from json.load(f)
    else:  # csv / tsv
        delim = "\t" if ext == ".tsv" else ","
        with open(path, encoding="utf-8", newline="") as f:
            yield from csv.DictReader(f, delimiter=delim)


class CEFRDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[cfg.label_col]).strip().upper()
            if lab not in LABEL2ID:          # drop A1/C2 or stray labels
                skipped += 1
                continue
            self.ex.append((str(row[cfg.text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, cfg.max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc

In [3]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
train_ds = CEFRDataset(cfg.train_path, tokenizer, cfg)
eval_ds  = CEFRDataset(cfg.eval_path,  tokenizer, cfg)
collator = DataCollatorWithPadding(tokenizer)

data/train.jsonl: kept 3797, skipped 0 out-of-scope
data/val.jsonl: kept 599, skipped 0 out-of-scope


### Double check that 1,024 max_length is enough to include all texts
#### It's well within limit

In [ ]:
# import numpy as np

# def token_lengths(path):
#     out = []
#     for row in read_rows(path):
#         if str(row[cfg.label_col]).strip().upper() not in LABEL2ID:
#             continue                      # same filter as CEFRDataset
#         out.append(len(tokenizer(str(row[cfg.text_col]), truncation=False)["input_ids"]))
#     return out

# all_lens = []
# for name, path in [("train", cfg.train_path), ("val", cfg.eval_path), ("test", cfg.test_path)]:
#     lens = token_lengths(path)
#     all_lens += lens
#     print(f"{name:5s} n={len(lens):4d}  max={max(lens):4d}  p99={int(np.percentile(lens, 99)):4d}")

# print(f"\nglobal max = {max(all_lens)}  ->  set max_length >= this")

train n=3797  max= 626  p99= 422
val   n= 599  max= 621  p99= 407
test  n= 604  max= 463  p99= 365

global max = 626  ->  set max_length >= this


In [4]:
# --- model --------------------------------------------------------------
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = T5Gemma2OrdinalClassifier(
    cfg.model_id, num_classes=NUM_CLASSES, mode=cfg.mode, dtype=dtype,
)

if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=2 * cfg.lora_r, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        modules_to_save=["head"],   # always fully train the small head
    ))
    model.print_trainable_parameters()

Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

In [5]:
# --- trainer: mode-aware loss, shared metric panel ----------------------
class OrdinalTrainer(Trainer):
    def __init__(self, *a, mode="corn", **k):
        super().__init__(*a, **k)
        self.mode = mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        logits = out.logits
        if self.mode == "corn":
            loss = corn_loss(logits, labels.long(), NUM_CLASSES)
        else:
            loss = torch.nn.functional.cross_entropy(logits, labels.long())
        return (loss, out) if return_outputs else loss


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
    if cfg.mode == "corn":
        preds = corn_label_from_logits(logits).numpy()
    else:
        preds = logits.argmax(-1).numpy()
    return evaluate_predictions(eval_pred.label_ids, preds)   # identical to Gemma 4

In [6]:
# --- train --------------------------------------------------------------
args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",   # select the best checkpoint on QWK
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = OrdinalTrainer(
    model=model, args=args, mode=cfg.mode,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.350326,0.335609,0.864141,0.617696,0.459098,0.924875,0.292807,0.345834,0.289941,0.437764,0.459098,0.421680
2,0.220792,0.308881,0.880491,0.529215,0.519199,0.953255,0.618245,0.420667,0.421791,0.565576,0.519199,0.509481
3,0.113322,0.525463,0.854359,0.550918,0.512521,0.946578,0.441542,0.384794,0.385576,0.506588,0.512521,0.492874
4,0.034402,0.953602,0.890712,0.484140,0.555927,0.959933,0.599405,0.445105,0.446803,0.578053,0.555927,0.544561


TrainOutput(global_step=1900, training_loss=0.20310253419374164, metrics={'train_runtime': 68966.082, 'train_samples_per_second': 0.22, 'train_steps_per_second': 0.028, 'total_flos': 0.0, 'train_loss': 0.20310253419374164, 'epoch': 4.0})

In [9]:
# --- dev (val) metrics + predictions — safe to run on every condition ---
out = trainer.predict(eval_ds, metric_key_prefix="eval")   # eval_* keys, like before
logits = out.predictions
if isinstance(logits, tuple):
    logits = logits[0]
logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
y_pred = (corn_label_from_logits(logits).numpy()
          if cfg.mode == "corn" else logits.argmax(-1).numpy())
y_true = out.label_ids

print("best dev:", json.dumps(out.metrics, indent=2))
os.makedirs(cfg.output_dir, exist_ok=True)

with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(out.metrics, f, indent=2)

import csv
with open(os.path.join(cfg.output_dir, "dev_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["true_idx", "pred_idx", "true_label", "pred_label"])
    for t, p in zip(y_true, y_pred):
        w.writerow([int(t), int(p), LEVELS[int(t)], LEVELS[int(p)]])

best dev: {
  "eval_loss": 0.9536022543907166,
  "eval_qwk": 0.8907117216809263,
  "eval_mae": 0.48414023372287146,
  "eval_accuracy": 0.5559265442404007,
  "eval_adjacent_accuracy": 0.9599332220367279,
  "eval_precision_macro": 0.5994051930643616,
  "eval_recall_macro": 0.4451048656341012,
  "eval_f1_macro": 0.4468025495349883,
  "eval_precision_weighted": 0.5780532757433118,
  "eval_recall_weighted": 0.5559265442404007,
  "eval_f1_weighted": 0.5445606655177622,
  "eval_runtime": 247.454,
  "eval_samples_per_second": 2.421,
  "eval_steps_per_second": 0.303
}


# CORN Confusion Matrix

In [10]:
# Cell: confusion matrix from saved predictions (val set)
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

run_dir = cfg.output_dir   # point at any condition's dir, e.g. "runs/t5gemma2_corn"
df = pd.read_csv(os.path.join(run_dir, "dev_predictions.csv"))
y_true, y_pred = df["true_idx"].to_numpy(), df["pred_idx"].to_numpy()

n = len(LEVELS)
counts = confusion_matrix(y_true, y_pred, labels=list(range(n)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(n):
    for j in range(n):
        annot[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-normalized"),
))
fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="True", autorange="reversed")
fig.update_layout(
    title=dict(text=f"T5Gemma 2 — {os.path.basename(run_dir)} (val)"),
    font=dict(family="Arial", size=16, color="black"),
    width=620, height=560,
    margin=dict(l=80, r=100, t=100, b=80),
)
fig.show()

# CORN Histogram vs. True Labels

In [11]:
# Cell: ridgeline of label distributions from saved predictions (val)
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# (display name, run dir) — uncomment the Nominal line once that run exists
conditions = [
    ("CORN", "runs/t5gemma2_corn"),
    # ("Nominal", "runs/t5gemma2_nominal"),
]

# true labels are shared across conditions (same val set, same order) -> read once
first_df = pd.read_csv(os.path.join(conditions[0][1], "dev_predictions.csv"))
ridge_rows = [("True labels", first_df["true_label"])]
for name, run_dir in conditions:
    df = pd.read_csv(os.path.join(run_dir, "dev_predictions.csv"))
    ridge_rows.append((name, df["pred_label"]))

idx = {lvl: i for i, lvl in enumerate(LEVELS)}
n_levels = len(LEVELS)

def level_proportions(series):
    s = series[series.isin(LEVELS)]
    counts = s.map(idx).value_counts().reindex(range(n_levels), fill_value=0)
    total = counts.sum()
    return (counts / total).to_numpy() if total else counts.to_numpy().astype(float)

def row_color(name):
    if name == "True labels": return "rgba(90,90,90,0.55)"
    if name == "CORN":        return "rgba(70,110,170,0.55)"
    if name == "Nominal":     return "rgba(200,90,80,0.55)"
    return "rgba(120,120,120,0.55)"

spacing = 1.0
scale   = 1.3

fig = go.Figure()
n_rows = len(ridge_rows)

for i, (name, series) in enumerate(ridge_rows):
    base = (n_rows - 1 - i) * spacing
    p = level_proportions(series)

    xs, ys = [], []
    for j in range(n_levels):
        xs += [j - 0.5, j + 0.5]
        ys += [base + scale * p[j]] * 2
    poly_x = [-0.5] + xs + [n_levels - 0.5]
    poly_y = [base] + ys + [base]

    fig.add_trace(go.Scatter(
        x=poly_x, y=poly_y, fill="toself", mode="lines",
        line=dict(color="black", width=1), fillcolor=row_color(name),
        hoverinfo="skip", showlegend=False,
    ))

    mean_idx = float(np.dot(np.arange(n_levels), p))
    fig.add_trace(go.Scatter(
        x=[mean_idx], y=[base], mode="markers",
        marker=dict(color="black", size=10, symbol="line-ns-open"),
        hoverinfo="skip", showlegend=False,
    ))

true_mean = float(np.dot(np.arange(n_levels), level_proportions(ridge_rows[0][1])))
fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

fig.update_layout(
    title="CEFR-level distributions — T5Gemma 2 (val)",
    xaxis=dict(tickmode="array", tickvals=list(range(n_levels)), ticktext=LEVELS,
               title="CEFR level", range=[-0.6, n_levels - 0.4]),
    yaxis=dict(tickmode="array",
               tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
               ticktext=[name for name, _ in ridge_rows],
               showgrid=False, zeroline=False),
    height=105 * n_rows + 120, width=900,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
)
fig.show()

# Baseline (Non-ordinal classifier)

In [12]:
# --- switch to nominal baseline -----------------------------------------
import gc
del trainer, model
gc.collect(); torch.cuda.empty_cache()

cfg.mode       = "nominal"
cfg.output_dir = "runs/t5gemma2_nominal"

### Run everything again

In [13]:
# --- model --------------------------------------------------------------
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = T5Gemma2OrdinalClassifier(
    cfg.model_id, num_classes=NUM_CLASSES, mode=cfg.mode, dtype=dtype,
)

if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=2 * cfg.lora_r, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        modules_to_save=["head"],   # always fully train the small head
    ))
    model.print_trainable_parameters()

Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

In [14]:
# --- train --------------------------------------------------------------
args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = OrdinalTrainer(
    model=model, args=args, mode=cfg.mode,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,1.392975,1.285518,0.844630,0.684474,0.437396,0.893155,0.316610,0.334526,0.263903,0.474747,0.437396,0.384327
2,0.997145,1.115735,0.867890,0.530885,0.525876,0.948247,0.522648,0.403411,0.382382,0.541848,0.525876,0.497962
3,0.582768,1.460692,0.858863,0.532554,0.532554,0.944908,0.457364,0.382627,0.371968,0.529163,0.532554,0.505777
4,0.164641,3.714126,0.878482,0.509182,0.542571,0.949917,0.476883,0.409423,0.398760,0.549771,0.542571,0.525023


TrainOutput(global_step=1900, training_loss=0.8938519334793091, metrics={'train_runtime': 99346.636, 'train_samples_per_second': 0.153, 'train_steps_per_second': 0.019, 'total_flos': 0.0, 'train_loss': 0.8938519334793091, 'epoch': 4.0})

In [15]:
# --- dev (val) metrics + predictions — safe to run on every condition ---
out = trainer.predict(eval_ds, metric_key_prefix="eval")   # eval_* keys, like before
logits = out.predictions
if isinstance(logits, tuple):
    logits = logits[0]
logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
y_pred = (corn_label_from_logits(logits).numpy()
          if cfg.mode == "corn" else logits.argmax(-1).numpy())
y_true = out.label_ids

print("best dev:", json.dumps(out.metrics, indent=2))
os.makedirs(cfg.output_dir, exist_ok=True)

with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(out.metrics, f, indent=2)

import csv
with open(os.path.join(cfg.output_dir, "dev_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["true_idx", "pred_idx", "true_label", "pred_label"])
    for t, p in zip(y_true, y_pred):
        w.writerow([int(t), int(p), LEVELS[int(t)], LEVELS[int(p)]])

best dev: {
  "eval_loss": 3.7141261100769043,
  "eval_qwk": 0.8784819521527945,
  "eval_mae": 0.5091819699499165,
  "eval_accuracy": 0.5425709515859767,
  "eval_adjacent_accuracy": 0.9499165275459098,
  "eval_precision_macro": 0.47688305389840613,
  "eval_recall_macro": 0.4094228199693495,
  "eval_f1_macro": 0.3987597328467559,
  "eval_precision_weighted": 0.5497707706784847,
  "eval_recall_weighted": 0.5425709515859767,
  "eval_f1_weighted": 0.5250230814601109,
  "eval_runtime": 510.8941,
  "eval_samples_per_second": 1.172,
  "eval_steps_per_second": 0.147
}


# Confusion matrices CORN vs. Nominal

In [16]:
# Cell: side-by-side confusion matrices (val) — CORN vs nominal
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

# (subplot title, run dir) — add a third tuple later for the fine-tuned decoder
panels = [
    ("CORN",    "runs/t5gemma2_corn"),
    ("Nominal", "runs/t5gemma2_nominal"),
]

n = len(LEVELS)

def counts_norm(run_dir):
    df = pd.read_csv(os.path.join(run_dir, "dev_predictions.csv"))
    y_true, y_pred = df["true_idx"].to_numpy(), df["pred_idx"].to_numpy()
    counts = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    row_sums = counts.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))
    return counts, norm

def annot(counts, norm):
    a = np.empty_like(counts, dtype=object)
    for i in range(n):
        for j in range(n):
            a[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
    return a

fig = make_subplots(
    rows=1, cols=len(panels),
    subplot_titles=[name for name, _ in panels],
    horizontal_spacing=0.12,
)
fig.update_layout(coloraxis=dict(
    colorscale="Greys", cmin=0, cmax=1,
    colorbar=dict(title="Row-normalized"),
))

for k, (name, run_dir) in enumerate(panels):
    counts, norm = counts_norm(run_dir)
    fig.add_trace(
        go.Heatmap(
            z=norm, x=LEVELS, y=LEVELS,
            text=annot(counts, norm), texttemplate="%{text}",
            hoverongaps=False, coloraxis="coloraxis",
        ),
        row=1, col=k + 1,
    )
    fig.update_xaxes(title_text="Predicted", row=1, col=k + 1)
    fig.update_yaxes(title_text="True", autorange="reversed", row=1, col=k + 1)

fig.update_layout(
    title=dict(text="T5Gemma 2 — confusion matrices (val)"),
    font=dict(family="Arial", size=14, color="black"),
    width=560 * len(panels), height=560,
    margin=dict(l=80, r=100, t=120, b=80),
)
fig.show()

# Histograms (including CORN and Nominal)

In [17]:
# Cell: ridgeline of label distributions from saved predictions (val)
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# (display name, run dir) — uncomment the Nominal line once that run exists
conditions = [
    ("CORN", "runs/t5gemma2_corn"),
    ("Nominal", "runs/t5gemma2_nominal"),
]

# true labels are shared across conditions (same val set, same order) -> read once
first_df = pd.read_csv(os.path.join(conditions[0][1], "dev_predictions.csv"))
ridge_rows = [("True labels", first_df["true_label"])]
for name, run_dir in conditions:
    df = pd.read_csv(os.path.join(run_dir, "dev_predictions.csv"))
    ridge_rows.append((name, df["pred_label"]))

idx = {lvl: i for i, lvl in enumerate(LEVELS)}
n_levels = len(LEVELS)

def level_proportions(series):
    s = series[series.isin(LEVELS)]
    counts = s.map(idx).value_counts().reindex(range(n_levels), fill_value=0)
    total = counts.sum()
    return (counts / total).to_numpy() if total else counts.to_numpy().astype(float)

def row_color(name):
    if name == "True labels": return "rgba(90,90,90,0.55)"
    if name == "CORN":        return "rgba(70,110,170,0.55)"
    if name == "Nominal":     return "rgba(200,90,80,0.55)"
    return "rgba(120,120,120,0.55)"

spacing = 1.0
scale   = 1.3

fig = go.Figure()
n_rows = len(ridge_rows)

for i, (name, series) in enumerate(ridge_rows):
    base = (n_rows - 1 - i) * spacing
    p = level_proportions(series)

    xs, ys = [], []
    for j in range(n_levels):
        xs += [j - 0.5, j + 0.5]
        ys += [base + scale * p[j]] * 2
    poly_x = [-0.5] + xs + [n_levels - 0.5]
    poly_y = [base] + ys + [base]

    fig.add_trace(go.Scatter(
        x=poly_x, y=poly_y, fill="toself", mode="lines",
        line=dict(color="black", width=1), fillcolor=row_color(name),
        hoverinfo="skip", showlegend=False,
    ))

    mean_idx = float(np.dot(np.arange(n_levels), p))
    fig.add_trace(go.Scatter(
        x=[mean_idx], y=[base], mode="markers",
        marker=dict(color="black", size=10, symbol="line-ns-open"),
        hoverinfo="skip", showlegend=False,
    ))

true_mean = float(np.dot(np.arange(n_levels), level_proportions(ridge_rows[0][1])))
fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

fig.update_layout(
    title="CEFR-level distributions — T5Gemma 2 (val)",
    xaxis=dict(tickmode="array", tickvals=list(range(n_levels)), ticktext=LEVELS,
               title="CEFR level", range=[-0.6, n_levels - 0.4]),
    yaxis=dict(tickmode="array",
               tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
               ticktext=[name for name, _ in ridge_rows],
               showgrid=False, zeroline=False),
    height=105 * n_rows + 120, width=900,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
)
fig.show()

## Compare metrics in two conditions 
### (rounded to 2 decimal places)

In [18]:
# Cell: summary table of dev metrics — CORN vs nominal
import os, json
import pandas as pd

conditions = [
    ("CORN",    "runs/t5gemma2_corn"),
    ("Nominal", "runs/t5gemma2_nominal"),
]

# Trainer bookkeeping + non-comparable loss -> keep out of the table
drop = {"loss", "runtime", "samples_per_second", "steps_per_second", "epoch"}

cols = {}
for name, run_dir in conditions:
    with open(os.path.join(run_dir, "dev_metrics.json")) as f:
        m = json.load(f)
    cols[name] = {
        (k[5:] if k.startswith("eval_") else k): v
        for k, v in m.items()
        if (k[5:] if k.startswith("eval_") else k) not in drop
    }

table = pd.DataFrame(cols)

# put the primary metrics first; any others fall in after, nothing is dropped
preferred = ["qwk", "mae", "exact_acc", "adjacent_acc",
             "f1_macro", "f1_weighted",
             "precision_macro", "recall_macro",
             "precision_weighted", "recall_weighted"]
order = [m for m in preferred if m in table.index] + \
        [m for m in table.index if m not in preferred]
table = table.loc[order].round(2)

table.to_csv("runs/dev_summary.csv")
table

,CORN,Nominal
qwk,0.89,0.88
mae,0.48,0.51
f1_macro,0.45,0.40
f1_weighted,0.54,0.53
precision_macro,0.60,0.48
recall_macro,0.45,0.41
precision_weighted,0.58,0.55
recall_weighted,0.56,0.54
accuracy,0.56,0.54
adjacent_accuracy,0.96,0.95


# Model finetuning time:
### 1149 minutes for CORN (19.15 hours)
### 1656 minutes for nominal (27.6 hours)


# Test data for very end only!

In [ ]:
# from transformers import AutoTokenizer
# # change corn to nominal to load the other model
# ckpt = "runs/t5gemma2_corn/checkpoint-XXXX"   # the best_model_checkpoint from trainer_state.json
# # change corn to nominal to load the other model
# model = T5Gemma2OrdinalClassifier(cfg.model_id, num_classes=NUM_CLASSES, mode="corn", dtype=torch.bfloat16)
# model.load_state_dict(load_file(f"{ckpt}/model.safetensors"))  # from safetensors.torch import load_file
# tokenizer = AutoTokenizer.from_pretrained(ckpt)

In [ ]:
# # --- FINAL TEST — RUN ONCE, AFTER CHOOSING THE FINAL CONDITION ----------
# test_ds  = CEFRDataset(cfg.test_path, tokenizer, cfg)
# test_out = trainer.predict(test_ds)
# print(json.dumps(test_out.metrics, indent=2))

# os.makedirs(cfg.output_dir, exist_ok=True)
# with open(os.path.join(cfg.output_dir, "test_metrics.json"), "w") as f:
#     json.dump(test_out.metrics, f, indent=2)